# Sonar : Classification Mines vs Rochers

**Objectif :** Construire un pipeline de Machine Learning complet pour classifier des signaux sonar en deux catégories :
- **M** — Mine (cylindre métallique)
- **R** — Roche (roche cylindrique)

Dataset original de Gorman & Sejnowski (1988) — 208 échantillons, 60 features acoustiques.

---
## 0. Imports & Configuration globale

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, f1_score, accuracy_score
)

# Classifieurs
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Répertoire de sortie pour les graphiques
PLOTS_DIR = os.path.join('outputs', 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

# Style global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.bbox': 'tight'})

print('Environnement prêt. Répertoire de plots :', PLOTS_DIR)

---
## 1. Définition du Problème

**Classification binaire supervisée.**

Un sonar émet un signal acoustique (chirp FM) et enregistre l'écho renvoyé par un objet sous-marin. Chaque échantillon contient 60 valeurs d'énergie dans 60 bandes de fréquences différentes, normalisées entre 0 et 1. La tâche est de déterminer si l'objet est :
- Une **Mine** (`M`) — cylindre métallique
- Une **Roche** (`R`) — roche de forme similaire

**Métriques cibles :** Accuracy, F1-score (macro), AUC-ROC.

---
## 2. Acquisition des Données

Détection automatique du chemin (Kaggle vs. local).

In [ ]:
# --- Détection automatique du chemin ---
KAGGLE_PATH = '/kaggle/input/connectionist-bench-sonar-mines-vs-rocks/sonar.all-data.csv'
LOCAL_PATH  = os.path.join('..', 'Documents', 'Sonar-dataset', 'sonar.all-data.csv')

if os.path.exists(KAGGLE_PATH):
    DATA_PATH = KAGGLE_PATH
    print('Exécution sur Kaggle — chemin Kaggle utilisé.')
elif os.path.exists(LOCAL_PATH):
    DATA_PATH = LOCAL_PATH
    print('Exécution en local — chemin local utilisé.')
else:
    # Fallback : même répertoire que le notebook
    DATA_PATH = 'sonar.all-data.csv'
    print('Chemin fallback :', DATA_PATH)

# --- Noms de colonnes : F1 à F60 + Label ---
feature_names = [f'F{i}' for i in range(1, 61)]
col_names = feature_names + ['Label']

# --- Chargement ---
df = pd.read_csv(DATA_PATH, header=None, names=col_names)

print(f'\nDimensions du dataset : {df.shape}')
print(f'Colonnes : {df.columns.tolist()[:5]} ... {df.columns.tolist()[-3:]}')
df.head(3)

---
## 3. Nettoyage des Données

Vérification des valeurs manquantes, encodage des étiquettes, découpage stratifié 80/20, normalisation.

In [ ]:
# --- 3.1 Vérification des valeurs nulles ---
null_count = df.isnull().sum().sum()
print(f'Valeurs manquantes totales : {null_count}')
assert null_count == 0, 'Des valeurs manquantes ont été détectées !'

# --- 3.2 Types de données ---
print('\nTypes de données :')
print(df.dtypes.value_counts())

# --- 3.3 Encodage des étiquettes : M=1, R=0 ---
le = LabelEncoder()
df['Target'] = le.fit_transform(df['Label'])   # R→0, M→1
print('\nEncodage :', dict(zip(le.classes_, le.transform(le.classes_))))

# --- 3.4 Séparation features / cible ---
X = df[feature_names].values
y = df['Target'].values

# --- 3.5 Découpage stratifié 80/20 ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'\nTrain : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Distribution train → M:{y_train.sum()} / R:{(y_train==0).sum()}')
print(f'Distribution test  → M:{y_test.sum()} / R:{(y_test==0).sum()}')

# --- 3.6 Normalisation StandardScaler (fit sur train uniquement) ---
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('\nNormalisation appliquée (μ≈0, σ≈1 sur le train).')

---
## 4. Analyse Exploratoire des Données (EDA)

Distribution des classes, profils moyens des signaux, corrélations, boxplots.

In [ ]:
# ---- 4.1 Distribution des classes ----------------------------------------
fig, ax = plt.subplots(figsize=(5, 4))
class_counts = df['Label'].value_counts()
bars = ax.bar(class_counts.index, class_counts.values,
              color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.5)
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontweight='bold')
ax.set_title('Distribution des classes', fontsize=14, fontweight='bold')
ax.set_xlabel('Classe')
ax.set_ylabel('Nombre d\'échantillons')
ax.set_ylim(0, class_counts.max() + 15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '01_class_distribution.png'))
plt.show()
print('Ratio M/R :', round(class_counts['M'] / class_counts['R'], 3))

In [ ]:
# ---- 4.2 Profils moyens des signaux (Mine vs Roche) -----------------------
df_feat = df[feature_names + ['Label']]
mean_M = df_feat[df_feat['Label'] == 'M'][feature_names].mean()
mean_R = df_feat[df_feat['Label'] == 'R'][feature_names].mean()

x_idx = range(1, 61)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(x_idx, mean_M.values, label='Mine (M)', color='#DD8452', linewidth=2)
ax.plot(x_idx, mean_R.values, label='Roche (R)', color='#4C72B0', linewidth=2)
ax.fill_between(x_idx, mean_M.values, mean_R.values, alpha=0.15, color='grey')
ax.set_title('Profil moyen du signal sonar par classe', fontsize=14, fontweight='bold')
ax.set_xlabel('Bande de fréquence (F1 → F60)')
ax.set_ylabel('Énergie moyenne normalisée')
ax.legend()
ax.set_xticks([1, 10, 20, 30, 40, 50, 60])
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '02_mean_signal_profiles.png'))
plt.show()

In [ ]:
# ---- 4.3 Heatmap de corrélation (sous-ensemble F1-F20 pour lisibilité) ----
corr_cols = feature_names[:20]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=False, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.4, ax=ax
)
ax.set_title('Matrice de corrélation — Features F1 à F20', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '03_correlation_heatmap.png'))
plt.show()

In [ ]:
# ---- 4.4 Boxplots — Top 12 features les plus discriminantes ---------------
# Sélection des 12 features avec la plus grande différence |mean_M - mean_R|
diff = (mean_M - mean_R).abs().sort_values(ascending=False)
top12 = diff.index[:12].tolist()

df_melt = df[top12 + ['Label']].melt(id_vars='Label', var_name='Feature', value_name='Valeur')

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()
for i, feat in enumerate(top12):
    sns.boxplot(
        data=df_melt[df_melt['Feature'] == feat],
        x='Label', y='Valeur',
        palette={'M': '#DD8452', 'R': '#4C72B0'},
        ax=axes[i]
    )
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

fig.suptitle('Boxplots — Top 12 features discriminantes (Mine vs Roche)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '04_boxplots_top12_features.png'))
plt.show()

---
## 5. Sélection et Comparaison des Modèles

Comparaison de 7 classifieurs via **StratifiedKFold (k=10)** avec `StandardScaler` intégré dans chaque pipeline.

Classifieurs évalués :
1. Régression Logistique
2. K-Nearest Neighbors
3. SVM à noyau RBF
4. Arbre de Décision
5. Forêt Aléatoire
6. Gradient Boosting
7. Naïve Bayes Gaussien

In [ ]:
# --- Définition des modèles (avec scaler intégré via Pipeline) ---
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'KNN'                 : KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)'           : SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'Decision Tree'       : DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Naïve Bayes'         : GaussianNB(),
}

# --- StratifiedKFold CV-10 sur X_train_sc ---
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_sc, y_train, cv=skf, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<22} | CV Accuracy : {scores.mean():.4f} ± {scores.std():.4f}')

results_df = pd.DataFrame(cv_results).T
results_df.columns = [f'Fold_{i+1}' for i in range(10)]
results_df['Mean'] = results_df.mean(axis=1)
results_df['Std']  = results_df.std(axis=1)
results_df.sort_values('Mean', ascending=False)

In [ ]:
# --- Visualisation de la comparaison des modèles ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot des scores CV
plot_data = pd.DataFrame({k: v for k, v in cv_results.items()})
order = plot_data.mean().sort_values(ascending=False).index.tolist()

sns.boxplot(data=plot_data[order], orient='h', palette='muted', ax=axes[0])
axes[0].set_title('Distribution des scores CV-10 (Accuracy)', fontweight='bold')
axes[0].set_xlabel('Accuracy')
axes[0].axvline(x=0.8, color='red', linestyle='--', alpha=0.5, label='Seuil 80%')
axes[0].legend()

# Barplot des moyennes ± std
means = [cv_results[m].mean() for m in order]
stds  = [cv_results[m].std()  for m in order]
colors = sns.color_palette('muted', n_colors=len(order))
bars = axes[1].barh(order, means, xerr=stds, color=colors,
                    edgecolor='white', capsize=5, height=0.6)
axes[1].set_title('Accuracy moyenne ± écart-type (CV-10)', fontweight='bold')
axes[1].set_xlabel('Accuracy')
axes[1].set_xlim(0.5, 1.05)
for bar, mean_val in zip(bars, means):
    axes[1].text(mean_val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{mean_val:.3f}', va='center', fontsize=9)

plt.suptitle('Comparaison des 7 classifieurs — Sonar Dataset',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '05_model_comparison.png'))
plt.show()

# Identification du meilleur modèle
best_model_name = max(cv_results, key=lambda k: cv_results[k].mean())
print(f'\nMeilleur modèle : {best_model_name}')
print(f'Score CV moyen  : {cv_results[best_model_name].mean():.4f}')

---
## 6. Résultats & Optimisation du Meilleur Modèle

**GridSearchCV** sur le meilleur modèle identifié, puis évaluation complète sur le jeu de test :
matrice de confusion, courbe ROC, importances des features.

In [ ]:
# --- 6.1 GridSearchCV sur le meilleur modèle ---

# Grilles de paramètres par modèle
param_grids = {
    'Logistic Regression' : {'C': [0.01, 0.1, 1, 10, 100], 'solver': ['lbfgs', 'liblinear']},
    'KNN'                 : {'n_neighbors': [3, 5, 7, 9, 11], 'weights': ['uniform', 'distance']},
    'SVM (RBF)'           : {'C': [0.1, 1, 10, 50], 'gamma': ['scale', 'auto', 0.01, 0.001]},
    'Decision Tree'       : {'max_depth': [None, 5, 10, 15], 'min_samples_split': [2, 5, 10]},
    'Random Forest'       : {'n_estimators': [50, 100, 200], 'max_depth': [None, 5, 10],
                             'min_samples_split': [2, 5]},
    'Gradient Boosting'   : {'n_estimators': [50, 100, 200], 'learning_rate': [0.05, 0.1, 0.2],
                             'max_depth': [3, 5]},
    'Naïve Bayes'         : {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]},
}

base_model  = models[best_model_name]
param_grid  = param_grids[best_model_name]

grid_search = GridSearchCV(
    base_model, param_grid,
    cv=skf, scoring='accuracy', n_jobs=-1, refit=True
)
grid_search.fit(X_train_sc, y_train)

best_estimator = grid_search.best_estimator_
print(f'Meilleurs paramètres : {grid_search.best_params_}')
print(f'Meilleur score CV    : {grid_search.best_score_:.4f}')

In [ ]:
# --- 6.2 Évaluation sur le jeu de test ---
y_pred      = best_estimator.predict(X_test_sc)
y_pred_prob = best_estimator.predict_proba(X_test_sc)[:, 1]

test_accuracy = accuracy_score(y_test, y_pred)
test_f1       = f1_score(y_test, y_pred, average='macro')
fpr, tpr, _   = roc_curve(y_test, y_pred_prob)
test_auc      = auc(fpr, tpr)

print(f'Test Accuracy  : {test_accuracy:.4f}')
print(f'Test F1-score  : {test_f1:.4f}')
print(f'Test AUC-ROC   : {test_auc:.4f}')
print('\nRapport de classification :')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# --- 6.3 Matrice de confusion + Courbe ROC ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_,
    linewidths=1, ax=axes[0]
)
axes[0].set_title(f'Matrice de Confusion\n{best_model_name}', fontweight='bold')
axes[0].set_xlabel('Prédit')
axes[0].set_ylabel('Réel')

# Courbe ROC
axes[1].plot(fpr, tpr, color='#DD8452', linewidth=2.5,
             label=f'AUC = {test_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aléatoire')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#DD8452')
axes[1].set_title(f'Courbe ROC\n{best_model_name}', fontweight='bold')
axes[1].set_xlabel('Taux de Faux Positifs (FPR)')
axes[1].set_ylabel('Taux de Vrais Positifs (TPR)')
axes[1].legend(loc='lower right')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '06_confusion_matrix_roc.png'))
plt.show()

In [ ]:
# --- 6.4 Importances des features (si disponible) ---
def get_feature_importances(estimator, feature_names):
    """Extrait les importances selon le type de modèle."""
    if hasattr(estimator, 'feature_importances_'):          # RF, GBM, DT
        return estimator.feature_importances_
    elif hasattr(estimator, 'coef_'):                        # LR, SVM linéaire
        return np.abs(estimator.coef_[0])
    else:
        return None

importances = get_feature_importances(best_estimator, feature_names)

if importances is not None:
    imp_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)
    top_n = 20
    top_imp = imp_series.head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = sns.color_palette('viridis', n_colors=top_n)
    bars = ax.barh(top_imp.index[::-1], top_imp.values[::-1],
                   color=colors, edgecolor='white')
    ax.set_title(f'Top {top_n} Features les plus importantes\n{best_model_name}',
                 fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '07_feature_importances.png'))
    plt.show()
    print(f'Top 5 features : {top_imp.index[:5].tolist()}')
else:
    print(f'Importances non disponibles pour {best_model_name}.')
    # Utiliser Random Forest comme proxy pour les importances
    print('Calcul via Random Forest (proxy)...')
    rf_proxy = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
    rf_proxy.fit(X_train_sc, y_train)
    imp_series = pd.Series(rf_proxy.feature_importances_, index=feature_names).sort_values(ascending=False)
    top_n = 20
    top_imp = imp_series.head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = sns.color_palette('viridis', n_colors=top_n)
    ax.barh(top_imp.index[::-1], top_imp.values[::-1], color=colors, edgecolor='white')
    ax.set_title(f'Top {top_n} Features les plus importantes (Random Forest proxy)',
                 fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '07_feature_importances_rf_proxy.png'))
    plt.show()
    print(f'Top 5 features (proxy RF) : {top_imp.index[:5].tolist()}')

---
## 7. Résumé Final

Synthèse des résultats obtenus sur le jeu de test après optimisation du meilleur modèle.

In [ ]:
# --- Tableau récapitulatif de tous les modèles (CV) ---
summary_cv = pd.DataFrame({
    'Modèle'   : list(cv_results.keys()),
    'CV Mean'  : [v.mean() for v in cv_results.values()],
    'CV Std'   : [v.std()  for v in cv_results.values()],
}).sort_values('CV Mean', ascending=False).reset_index(drop=True)

print('=' * 60)
print('       RÉSUMÉ — COMPARAISON CV (StratifiedKFold k=10)')
print('=' * 60)
print(summary_cv.to_string(index=False))
print()

print('=' * 60)
print(f'       MEILLEUR MODÈLE : {best_model_name}')
print('=' * 60)
print(f'  Paramètres optimaux  : {grid_search.best_params_}')
print(f'  Accuracy  (test)     : {test_accuracy:.4f}  ({test_accuracy*100:.2f}%)')
print(f'  F1-score  (macro)    : {test_f1:.4f}')
print(f'  AUC-ROC   (test)     : {test_auc:.4f}')
print('=' * 60)
print()
print('Graphiques sauvegardés dans :', PLOTS_DIR)
for fname in sorted(os.listdir(PLOTS_DIR)):
    print(' ', fname)